# MiniLLM Fast Inference & Architecture Deep Dive
This notebook enables you to **load a trained MiniLLM checkpoint from disk in seconds** and generate text interactively **without retraining**.

It also serves as a complete, block-by-block educational reference explaining every modern LLM architecture component:
1. **RMSNorm** (Root Mean Square Normalization)
2. **RoPE** (Rotary Position Embeddings)
3. **GQA** (Grouped-Query Attention)
4. **SwiGLU** (Swish-Gated Linear Units)
5. **TransformerBlock & MiniLLM** (Pre-LN & Weight Tying)
6. **BPETokenizer** (Restoring Merges from Disk)
7. **Autoregressive Sampling** (Temperature, Top-p, Softmax)
8. **Interactive Text Generation Playground**


In [1]:
import os
import re
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# Device detection (automatically utilizes your NVIDIA RTX 3050 GPU if available)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Using device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


---
## Module 1: RMSNorm (Root Mean Square Normalization)

### Why RMSNorm instead of LayerNorm?
Standard **LayerNorm** calculates both the mean and the variance of activations:
$$\text{LN}(x) = \frac{x - \mu}{\sigma} \cdot \gamma + \beta$$

However, research (Zhang & Sennrich, 2019; LLaMA, Gemma, Mistral) showed that **the mean-centering operation ($x - \mu$) contributes very little to training stability**. Most of the benefit comes from scaling by the root mean square.

**RMSNorm** removes the mean calculation entirely:
$$\text{RMS}(x) = \sqrt{\frac{1}{d} \sum_{i=1}^d x_i^2 + \epsilon}$$
$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \odot \gamma$$

### Benefits:
- **~10-20% faster** than standard LayerNorm because it avoids computing the mean and subtracting it.
- Has no bias parameter ($\beta$), saving parameters.
- Maintains training stability just as effectively as LayerNorm.


In [2]:
class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization.
    Normalizes activations across the last dimension by their root mean square.
    """
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # Learnable scale parameter gamma (initialized to ones)
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch, seq_len, dim)
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight

# Quick sanity check
demo_norm = RMSNorm(dim=256)
x_demo = torch.randn(2, 4, 256)
out_norm = demo_norm(x_demo)
print("RMSNorm input shape :", x_demo.shape)
print("RMSNorm output shape:", out_norm.shape)


RMSNorm input shape : torch.Size([2, 4, 256])
RMSNorm output shape: torch.Size([2, 4, 256])


---
## Module 2: RoPE (Rotary Position Embeddings)

### Why RoPE instead of Absolute Positional Embeddings?
In early models like GPT-2, positional information was added via learned absolute position vectors: $x = \text{tok\_emb} + \text{pos\_emb}$.
This had major flaws:
1. It does not generalize well to sequence lengths longer than seen in training.
2. It represents position *absolutely* ($pos=10$) rather than *relatively* (token A is 3 steps ahead of token B).

### How Rotary Embeddings Work (Su et al., 2021)
RoPE encodes position directly into the Query and Key vectors by **rotating** pairs of coordinates in 2D complex planes by an angle proportional to their sequence position $m$:
$$R_{\Theta, m}^d = \text{diag}\left(R_{\theta_1, m}, R_{\theta_2, m}, \dots, R_{\theta_{d/2}, m}\right)$$
where $\theta_i = 10000^{-2(i-1)/d}$.

When Query $q_m$ at position $m$ interacts with Key $k_n$ at position $n$ via dot product:
$$(R_m q)^T (R_n k) = q^T R_{n-m} k$$
The attention score **naturally depends only on the relative distance $(m - n)$!**

### In Our Code:
- `precompute_rope_freqs()` calculates the sinusoidal rotation angles for every position up to `max_seq_len`.
- `apply_rope()` rotates consecutive coordinate pairs $(x_1, x_2)$ using standard 2D rotation formulas:
  $$x_1' = x_1 \cos(\theta) - x_2 \sin(\theta)$$
  $$x_2' = x_1 \sin(\theta) + x_2 \cos(\theta)$$


In [3]:
def precompute_rope_freqs(head_dim: int, max_seq_len: int, base: float = 10000.0):
    """
    Precomputes cosine and sine angles for positional rotation.
    """
    # Frequency bands for each coordinate pair: shape (head_dim // 2,)
    freqs = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    # Positions in context: shape (max_seq_len,)
    positions = torch.arange(max_seq_len).float()
    # Outer product gives angles for all (pos, freq) combinations: (max_seq_len, head_dim // 2)
    angles = torch.outer(positions, freqs)
    return torch.cos(angles), torch.sin(angles)

def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    Applies 2D rotation to coordinate pairs along the head dimension.
    x shape: (batch, n_heads, seq_len, head_dim)
    """
    seq_len = x.shape[2]
    # Slice angles to current sequence length
    cos = cos[:seq_len].unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, head_dim // 2)
    sin = sin[:seq_len].unsqueeze(0).unsqueeze(0)

    # Split head_dim into even and odd indices
    x1 = x[..., ::2]   # Even indices (x-coordinates)
    x2 = x[..., 1::2]  # Odd indices (y-coordinates)

    # 2D planar rotation:
    # x1' = x1 * cos - x2 * sin
    # x2' = x1 * sin + x2 * cos
    out1 = x1 * cos - x2 * sin
    out2 = x1 * sin + x2 * cos

    # Interleave back to original head_dim
    return torch.stack([out1, out2], dim=-1).flatten(-2)

# Quick sanity check
c, s = precompute_rope_freqs(head_dim=32, max_seq_len=256)
demo_q = torch.randn(1, 8, 10, 32)
demo_q_rot = apply_rope(demo_q, c, s)
print("RoPE applied: input shape", demo_q.shape, "-> rotated shape", demo_q_rot.shape)


RoPE applied: input shape torch.Size([1, 8, 10, 32]) -> rotated shape torch.Size([1, 8, 10, 32])


---
## Module 3: GQA (Grouped-Query Attention)

### The KV-Cache Memory Bottleneck
During autoregressive text generation, every token generated requires storing and reading all previous Key and Value vectors (the **KV-Cache**). 
- In **Multi-Head Attention (MHA)**, if we have 8 Query heads, we also have 8 Key heads and 8 Value heads. Reading 8 KV heads on every step consumes huge memory bandwidth.
- In **Multi-Query Attention (MQA)**, there is only 1 Key head and 1 Value head shared across all 8 Query heads. While very fast, quality can drop.
- In **Grouped-Query Attention (GQA)** (Ainslie et al., 2023; LLaMA 2/3, Mistral), we group Query heads. For example:
  - 8 Query heads
  - 2 Key/Value heads
  - Each KV head is shared by 4 Query heads (ratio 4:1)

```
Multi-Head (MHA)        Grouped-Query (GQA)       Multi-Query (MQA)
Q1 Q2 Q3 Q4 Q5 Q6 Q7 Q8    Q1 Q2 Q3 Q4  Q5 Q6 Q7 Q8    Q1 Q2 Q3 Q4 Q5 Q6 Q7 Q8
 |  |  |  |  |  |  |  |      \  /  \  /    \  /  \  /       \   \   |   /   /
K1 K2 K3 K4 K5 K6 K7 K8        K1-2         K3-4                 K1
```

### In Our Code:
- `repeat_kv()` expands each KV head `n_rep` times (e.g. 4x) so standard batched matrix multiplication can be used for attention scores: `q @ k.transpose(-2, -1)`.
- Causal masking (`torch.triu`) ensures tokens can only attend to previous tokens.


In [4]:
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    Repeats KV heads along a new dimension when n_heads > n_kv_heads.
    x shape: (batch, n_kv_heads, seq_len, head_dim)
    output:  (batch, n_heads, seq_len, head_dim)
    """
    if n_rep == 1:
        return x
    b, n_kv, seq, hd = x.shape
    return (x[:, :, None, :, :]
            .expand(b, n_kv, n_rep, seq, hd)
            .reshape(b, n_kv * n_rep, seq, hd))

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.head_dim = d_model // n_heads

        # Linear projections for Q, K, V and Output
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(n_heads * self.head_dim, d_model, bias=False)
        self.dropout = dropout

    def forward(self, x: torch.Tensor, rope_cos: torch.Tensor, rope_sin: torch.Tensor) -> torch.Tensor:
        b, seq, _ = x.shape

        # 1. Project inputs to Q, K, V
        q = self.q_proj(x).view(b, seq, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, seq, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, seq, self.n_kv_heads, self.head_dim).transpose(1, 2)

        # 2. Apply RoPE to Query and Key
        q = apply_rope(q, rope_cos, rope_sin)
        k = apply_rope(k, rope_cos, rope_sin)

        # 3. Repeat KV heads to match Q heads count
        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        # 4. Scaled Dot-Product Attention with causal upper-triangular mask
        scale = 1.0 / math.sqrt(self.head_dim)
        scores = (q @ k.transpose(-2, -1)) * scale

        mask = torch.triu(torch.ones(seq, seq, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        if self.dropout > 0 and self.training:
            weights = F.dropout(weights, p=self.dropout)

        # 5. Weighted combination and output projection
        out = weights @ v
        out = out.transpose(1, 2).contiguous().view(b, seq, -1)
        return self.o_proj(out)

# Quick sanity check
demo_attn = GroupedQueryAttention(d_model=256, n_heads=8, n_kv_heads=2)
demo_out = demo_attn(x_demo, c, s)
print("GQA input:", x_demo.shape, "-> output:", demo_out.shape)


GQA input: torch.Size([2, 4, 256]) -> output: torch.Size([2, 4, 256])


---
## Module 4: SwiGLU (Swish-Gated Linear Units)

### Why SwiGLU instead of standard ReLU/GELU MLP?
In classic Transformers, the feed-forward network (FFN) is a simple 2-layer perceptron:
$$\text{FFN}(x) = \text{ReLU}(x W_1 + b_1) W_2 + b_2$$

Shazeer (2020) proposed **GLU (Gated Linear Unit) variants**. Among them, **SwiGLU** has proven empirically superior in virtually every modern open LLM (LLaMA, Mistral, PaLM, Gemma).

Instead of one upward projection, SwiGLU computes two parallel projections:
1. **Gate**: $W_{gate} x$, passed through the **SiLU (Swish)** non-linearity: $\text{SiLU}(z) = z \cdot \sigma(z)$.
2. **Up**: $W_{up} x$, a linear projection acting as content features.

The two are multiplied element-wise and projected back down:
$$\text{SwiGLU}(x) = \left( \text{SiLU}(x W_{gate}) \odot (x W_{up}) \right) W_{down}$$

The element-wise multiplication acts as a **dynamic feature filter** where the gate selectively modulates which dimensions pass through.


In [5]:
class SwiGLU(nn.Module):
    """
    Swish-Gated Linear Unit Feed-Forward Network.
    """
    def __init__(self, d_model: int, hidden_dim: int, dropout: float = 0.0):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden_dim, bias=False)
        self.w_up = nn.Linear(d_model, hidden_dim, bias=False)
        self.w_down = nn.Linear(hidden_dim, d_model, bias=False)
        self.dropout = dropout

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Gate path with SiLU activation: Swish(x * W_gate)
        gate = F.silu(self.w_gate(x))
        # Up path: (x * W_up)
        up = self.w_up(x)
        # Element-wise gating multiplied and projected back to d_model
        out = self.w_down(gate * up)
        if self.dropout > 0 and self.training:
            out = F.dropout(out, p=self.dropout)
        return out

# Quick sanity check
demo_ffn = SwiGLU(d_model=256, hidden_dim=680)
demo_ffn_out = demo_ffn(x_demo)
print("SwiGLU input:", x_demo.shape, "-> output:", demo_ffn_out.shape)


SwiGLU input: torch.Size([2, 4, 256]) -> output: torch.Size([2, 4, 256])


---
## Module 5: TransformerBlock & MiniLLM (Model Assembly & Weight Tying)

### Pre-LayerNorm Residual Design
Modern LLMs use **Pre-LN** (normalizing *before* each sub-layer rather than after):
$$x_{attn} = x + \text{Attention}(\text{RMSNorm}(x))$$
$$x_{out} = x_{attn} + \text{SwiGLU}(\text{RMSNorm}(x_{attn}))$$
This creates a clean, uninterrupted identity gradient highway directly from output to input.

### Weight Tying
Notice line 38 in `MiniLLM`:
```python
self.lm_head.weight = self.token_emb.weight
```
**Weight Tying** (Press & Wolf, 2017) shares the exact same weight matrix between the input token embeddings and the final linear prediction head.
- **Saves parameters**: For a vocabulary of size $V$ and hidden dimension $d$, this eliminates $V \times d$ parameters.
- **Semantic alignment**: A token vector that represents "meditation" at the input is directly aligned with predicting "meditation" at the output.


In [6]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int, ffn_hidden_dim: int, dropout: float = 0.0):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attention = GroupedQueryAttention(d_model, n_heads, n_kv_heads, dropout=dropout)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, ffn_hidden_dim, dropout=dropout)

    def forward(self, x: torch.Tensor, rope_cos: torch.Tensor, rope_sin: torch.Tensor) -> torch.Tensor:
        # Pre-LN attention block with residual connection
        x = x + self.attention(self.attn_norm(x), rope_cos, rope_sin)
        # Pre-LN FFN block with residual connection
        x = x + self.ffn(self.ffn_norm(x))
        return x

class MiniLLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, n_layers: int, n_heads: int, 
                 n_kv_heads: int, ffn_hidden_dim: int, max_seq_len: int, dropout: float = 0.0):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len

        # Token embedding lookup table
        self.token_emb = nn.Embedding(vocab_size, d_model)
        
        # Stack of Transformer blocks
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, n_kv_heads, ffn_hidden_dim, dropout=dropout)
            for _ in range(n_layers)
        ])
        
        # Final RMSNorm
        self.final_norm = RMSNorm(d_model)
        
        # Final language model prediction head
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight Tying: share embedding weights with lm_head
        self.lm_head.weight = self.token_emb.weight

        # Precompute RoPE rotary frequency buffers
        head_dim = d_model // n_heads
        rope_cos, rope_sin = precompute_rope_freqs(head_dim, max_seq_len)
        self.register_buffer("rope_cos", rope_cos)
        self.register_buffer("rope_sin", rope_sin)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None):
        x = self.token_emb(idx)
        for layer in self.layers:
            x = layer(x, self.rope_cos, self.rope_sin)
        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


---
## Module 6: BPE Tokenizer (Byte Pair Encoding Reconstructor)

### Why Must Tokenizer State Be Saved with the Model?
The token embedding matrix row $i$ corresponds specifically to token ID $i$.
If you retrain or re-extract BPE merges without preserving the exact ordered dictionary:
- Token ID `42` might mean `'the'` in training run 1.
- But in training run 2, ID `42` might mean `'ing'`.

By bundling `merges`, `token_to_id`, and `id_to_token` inside the checkpoint, `BPETokenizer` reconstructs the exact identical token mapping.


In [7]:
class BPETokenizer:
    """
    Reconstructs BPE tokenization using saved merges and vocabulary dictionaries.
    """
    def __init__(self, merges: dict, token_to_id: dict, id_to_token: dict):
        self.merges = merges
        self.token_to_id = token_to_id
        self.id_to_token = id_to_token
        self.unk_id = token_to_id.get("<unk>", 0)
        self.word_cache = {}

    def encode_word(self, w: str) -> list[int]:
        if w in self.word_cache:
            return self.word_cache[w]
        tokens = list(w)
        for pair, replacement in self.merges.items():
            if len(tokens) < 2:
                break
            i = 0
            new_tokens = []
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                    new_tokens.append(replacement)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens
        ids = [self.token_to_id.get(t, self.unk_id) for t in tokens]
        self.word_cache[w] = ids
        return ids

    def encode(self, s: str) -> list[int]:
        # Split on words and whitespace preserving exact original characters
        w_list = re.findall(r"\S+|\s+", s)
        out = []
        for w in w_list:
            out.extend(self.encode_word(w))
        return out

    def decode(self, ids: list[int]) -> str:
        return "".join(self.id_to_token.get(i, "<unk>") for i in ids)


---
## Module 7: Autoregressive Generation & Temperature Sampling

### How Text Generation Works
1. **Encode Prompt**: Convert text into token IDs $T = [t_1, t_2, \dots, t_k]$.
2. **Context Window**: Slice the last `max_seq_len` tokens: $T_{ctx} = T[-256:]$.
3. **Model Forward Pass**: Pass $T_{ctx}$ through the model to obtain logits for the final position.
4. **Temperature Scaling**: Divide logits by temperature $T$:
   $$z_i = \frac{\text{logit}_i}{T}$$
   - **Low Temperature ($T = 0.2 - 0.5$)**: Sharpens the distribution; the model becomes conservative, repetitive, and picks high-confidence tokens.
   - **Moderate Temperature ($T = 0.7 - 0.8$)**: Optimal balance between creativity, diversity, and coherence.
   - **High Temperature ($T \ge 1.0$)**: Flattens probabilities; increases randomness and creative leaps, but risks gibberish if too high.
5. **Sample**: Pass scaled logits through `softmax` and sample the next token using `torch.multinomial`.
6. **Append & Repeat**: Append the new token and repeat until `max_new_tokens` is reached.


In [8]:
@torch.no_grad()
def generate(model: MiniLLM, tokenizer: BPETokenizer, prompt: str, 
             max_new_tokens: int = 200, temperature: float = 0.8, 
             device: str = device) -> str:
    """
    Autoregressively generates text from a prompt using temperature sampling.
    """
    model.eval()
    tokens = tokenizer.encode(prompt)
    if not tokens:
        tokens = [0]
    tokens = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        # Truncate context to model's maximum sequence length
        context = tokens[:, -model.max_seq_len:]
        logits, _ = model(context)
        
        # Take logits for the last token position and scale by temperature
        logits = logits[:, -1, :] / max(temperature, 1e-5)
        probs = F.softmax(logits, dim=-1)
        
        # Sample next token from the probability distribution
        next_token = torch.multinomial(probs, num_samples=1)
        tokens = torch.cat([tokens, next_token], dim=1)

    return tokenizer.decode(tokens[0].tolist())


---
## Module 8: Checkpoint Loader

The function below inspects your folder for:
1. `mini_osho_llm_checkpoint.pt` (Trained on Osho's *The Book of Secrets*)
2. `mini_llm_bpe_checkpoint.pt` (Trained on Shakespeare)

It unpacks the weights, config, and tokenizer, initializing the model directly into GPU/CPU memory ready for generation.


In [9]:
def load_saved_model(checkpoint_path: str = None, device: str = device):
    """
    Loads a MiniLLM model and its BPE tokenizer from disk.
    """
    # Automatically locate checkpoint if not explicitly provided
    if checkpoint_path is None:
        candidates = ["mini_osho_llm_checkpoint.pt", "mini_llm_bpe_checkpoint.pt"]
        for c in candidates:
            if os.path.exists(c):
                checkpoint_path = c
                break

    if checkpoint_path is None or not os.path.exists(checkpoint_path):
        available = [f for f in os.listdir() if f.endswith(".pt")]
        raise FileNotFoundError(
            f"Checkpoint file not found!\n"
            f"Looking for: 'mini_osho_llm_checkpoint.pt' or 'mini_llm_bpe_checkpoint.pt'\n"
            f"Found .pt files in directory: {available}\n\n"
            f"To generate a checkpoint, run training first in 'mini_osho_llm.py' or 'mini-osho-llm.ipynb'."
        )

    print(f"Loading checkpoint '{checkpoint_path}' onto {device}...")
    ckpt = torch.load(checkpoint_path, map_location=device)

    # 1. Rebuild Tokenizer
    tokenizer = BPETokenizer(
        merges=ckpt["merges"],
        token_to_id=ckpt["token_to_id"],
        id_to_token=ckpt["id_to_token"]
    )
    print(f"  - Reconstructed BPE Tokenizer: {len(ckpt['token_to_id']):,} tokens ({len(ckpt['merges'])} merges)")

    # 2. Rebuild Model Architecture with saved config
    config = ckpt["config"]
    model = MiniLLM(**config).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"  - Model loaded: {config['n_layers']} layers, {config['n_heads']} heads, {total_params:,} parameters")
    print(f"Checkpoint '{checkpoint_path}' is ready for inference!\n")

    return model, tokenizer, config


In [10]:
# Execute this cell to load your saved checkpoint
# If you trained Osho, set: checkpoint_file = "mini_osho_llm_checkpoint.pt"
# If you trained Shakespeare, set: checkpoint_file = "mini_llm_bpe_checkpoint.pt"

try:
    model, tokenizer, config = load_saved_model()
except FileNotFoundError as e:
    print(e)


Loading checkpoint 'mini_osho_llm_checkpoint.pt' onto cuda...
  - Reconstructed BPE Tokenizer: 584 tokens (500 merges)
  - Model loaded: 4 layers, 8 heads, 2,896,128 parameters
Checkpoint 'mini_osho_llm_checkpoint.pt' is ready for inference!



---
## Module 9: Interactive Generation Playground

Now that the model and tokenizer are loaded in memory, you can generate text on any prompt instantly without retraining!

Experiment with:
- **Temperature ($0.3 - 1.2$)**: Observe how lower temperatures make the text focused and repetitive, while higher temperatures introduce variety.
- **Tokens ($50 - 500$)**: Length of continuation.
- **Different Prompts**: Dialogues, philosophical queries, or meditation techniques.


In [13]:
# Set your prompt and parameters here:
PROMPT = "life is good"
TEMPERATURE = 0.8
MAX_TOKENS = 250

if 'model' in locals():
    print("=" * 60)
    print(f"PROMPT: {PROMPT!r}")
    print(f"TEMPERATURE: {TEMPERATURE} | MAX NEW TOKENS: {MAX_TOKENS}")
    print("=" * 60 + "\n")
    
    result = generate(
        model=model, 
        tokenizer=tokenizer, 
        prompt=PROMPT, 
        max_new_tokens=MAX_TOKENS, 
        temperature=TEMPERATURE,
        device=device
    )
    print(result)
else:
    print("Model not loaded yet. Run Cell 18 after training a checkpoint!")


PROMPT: 'life is good'
TEMPERATURE: 0.8 | MAX NEW TOKENS: 250

life is good. 
Life is just a living flux. You are total in yourself because you are in love with someone. 
You are really in love with someone; you are not in love with someone, and 
you are filled with yourself. You are never involved; you are not in love. When someone 
loves you, you are not in love -- in love. 
The other love is a deep love; you are just a deep order or a love. Hate is the only love 
with you, and then love is a loving thing. You cannot love, and you are totally 
loving. You can go beyond love when he cannot 


In [12]:
# Compare generations across temperatures side-by-side
if 'model' in locals():
    test_prompt = "Meditation is not"
    for temp in [0.4, 0.7, 1.0]:
        print("-" * 50)
        print(f"Temperature: {temp}")
        print("-" * 50)
        print(generate(model, tokenizer, prompt=test_prompt, max_new_tokens=120, temperature=temp))
        print()


--------------------------------------------------
Temperature: 0.4
--------------------------------------------------
Meditation is not a meeting of the mind. The mind is a repetitive mind. The mind is 
not a meeting, it is a meeting meeting with the mind. The mind is a repetitive mind. It is 
not a meeting meeting you. You are not a meeting meeting through the

--------------------------------------------------
Temperature: 0.7
--------------------------------------------------
Meditation is not a problem. If you are dead, you are not defeated. You are just dead, you are defeated. 
You have never lived through it, and the transcendence has become a problem. You are 
destroyed. Your whole life is just a problem -- you are just a w

--------------------------------------------------
Temperature: 1.0
--------------------------------------------------
Meditation is not good, it is natural. It is trying to say, "This is not, that Christianity is fulfilled; you will 
have to give me to you